<div align="center">

# **LLM Visual Explorer V1**
## (PL, Aug26)

![LlmExpl — Visual Explorer](../images/LlmExpl_logo.png)

## Explorer l'espace d'un modèle de langage d'intelligence artificielle à travers un "trou de serrure"

</div>

Un mot/concept n'est pas un point isolé. Il possède une position dans un
espace sémantique de plusieurs centaines, voire milliers de dimensions.

---

### Cellules du Notebook

0. **Initialisation**

1. **Scénario**  
   Choix des concepts à explorer (de "animals" à "verbs").

2. **Embeddings**  
   Chargement du modèle et création des vecteurs sémantiques.

3. **Projection 3D + DataFrame**  
   Réduction de l'espace multidimensionnel et préparation des données.

4. **Visualisation 2D — "Carte sémantique"**  
   Exploration des concepts en 2D.

5. **Visualisation 3D — "Planétarium sémantique"**  
   Exploration des concepts en 3D.

6. **Similarités — Mesure des proximités sémantiques**  
   Classement (et archivage) des paires de concepts, du plus au moins similaire.

7. **Rapport final**  
   Synthèse et conservation des résultats de l'exploration.

8. **Messages**  
   En concluant vos explorations, avant de clore ce Notebook.

In [ ]:
#=======================================================
# 0 - Initialisation et import des libraries nécessaires
#=======================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from explorer.notebook_setup import *

In [ ]:
#=================================================
#  1 - Choix du scénario
#=================================================

from explorer.scenarios import scenario_selector
from explorer.display import display_scenario

selection = {}

scenario_selector(
    display_function=display_scenario,
    selection=selection
)

Output()

In [ ]:
# ==========================================================
# 2 Chargement du modèle et calcul des vecteurs sémantiques
# ==========================================================

# Recupération des données du scenario retenu
scenario = selection["scenario"]
concepts = selection["concepts"]
objects = selection["objects"]
icons = selection["icons"]
short_names = selection.get("short_names", {})

# Chargement du modèle & Calcul des embeddings / vecteurs sémantiques
embeddings = compute_embeddings(concepts)

# Récupération du modèle chargé
model = get_model()

# Affichage des informations (modèle & dimensiosn)
display_model(model)

MODÈLE GÉNÉRATIF — VARIATION D'ÉTAT INTERMÉDIAIRE

Modèle :
Nom Générique: PMG

Dimension de la représentation :
  896



In [13]:
#================================================
# 3 - Projection espace modèle => 3D (& DataFrame)
#================================================
# Projection
xyz, pca = compute_pca(
    embeddings
)
# Qualité / fidélité de la projection (max=100%)
display_projection(
    pca
)
# Préparation des données
df = create_dataframe(
    concepts,
    xyz,
)

PROJECTION 3D

La projection 3D conserve environ 66.7% de la variance des embeddings.



In [ ]:
#================================================
# 4 Semantic map 2D
#================================================
# Cette carte est une projection des concepts dans un espace mathématique.
# Deux points proches représentent des concepts dont les représentations 
# internes sont proches.

fig_map = plot_map(
    df=df,
    title="2D Semantic Map",
    icons=icons,
    short_names=short_names,
    show_labels=True,
    show_icons=False,
)

fig_map.show()

In [ ]:
#==============================================
# 5 Semantic Planetarium 3D
#================================================


fig = plot_scene(
    df=df,
    title="3D Semantic Planetarium",
    icons=icons,
    short_names=short_names,
    show_labels=True,
    show_icons=False,
)


fig.show()

In [15]:
#================================================
# 6   Classement des similarités
#================================================

from explorer.exports import save_similarities_csv

similarities = compute_similarity(embeddings)

similarity_pairs = rank_similarity_pairs(
    concepts,
    similarities
)

display_similarity_ranking(
    similarity_pairs,
    df,
    icons,
    top_n=5,
    bottom_n=5,
)

save_similarities_csv(
    similarity_pairs,
    scenario_name=selection["scenario_name"],
    model_alias=MODEL_ALIAS,
    model_name=MODEL_NAME,
)



🔗 SEMANTIC SIMILARITY vs 3D PROJECTION DISTANCE

Seuils indicatifs:
🟢 60–100 %   Similarité élevée
🟡 30–59 %    Similarité moyenne
🔴 0–29 %     Similarité faible

NB: L'échelle des similarités dépend du modèle. Les valeurs sont surtout comparables à l'intérieur d'un même modèle; le classement est plus significatif que la valeur absolue.

🟢 Les 5 voisins sémantiques
--------------------------------------------------------------------------------
     1. 🥖 Boulanger   ↔ ✈️ Pilote      ███████████████████  Sim: 95%  Dist: 0.6
     2. 💉 Infirmier   ↔ 🥖 Boulanger   ██████████████████   Sim: 94%  Dist: 0.6
     3. 👷 Ingénieur   ↔ 🔬 Chercheur   ██████████████████   Sim: 93%  Dist: 0.5
     4. 💉 Infirmier   ↔ ✈️ Pilote      ██████████████████   Sim: 93%  Dist: 0.7
     5. 🩺 Médecin     ↔ ⚖️ Avocat      ██████████████████   Sim: 93%  Dist: 2.4

--------------------------------------------------------------------------------
              ⋮ 35 paires intermédiaires masquées ⋮
------------------

WindowsPath('explorations/professions/PMG/similarities_2026-08-18_01-43-20-713160.csv')

In [ ]:
# ============================================================
# 6 TER — CONSTRUCTION PROGRESSIVE DU NOYAU SÉMANTIQUE
# ============================================================

from explorer.semantic_core import build_semantic_core

core_order, semantic_strength = build_semantic_core(
    embeddings,
    concepts
)

print("\n============================================================")
print("CONSTRUCTION PROGRESSIVE DU NOYAU")
print("============================================================")

for rank, concept in enumerate(core_order, start=1):

    strength = semantic_strength[concept]

    print(
        f"{rank:2d} | "
        f"{concept:12s} | "
        f"force = {strength:.3f}"
    )

In [ ]:
# ============================================================
# 5 BIS — PLANÉTARIUM AVEC FORCE SÉMANTIQUE
# ============================================================

fig = plot_scene(
    df,
    title="Planétarium — Force sémantique",
    icons=icons,
    short_names=short_names,
    semantic_strength=semantic_strength,
)
fig.show()

In [ ]:
#======================================================================
# 7 Display exploration report
#======================================================================

display_report(
    scenario=scenario,
    concepts=concepts,
    embedding_dimension=embeddings.shape[1],
    pca=pca,
    similarity_pairs=similarity_pairs,
    planetarium_figure=fig,
)


In [ ]:
# ================================================================
# 8. MESSAGES — Conclusion de l'exploration
# ================================================================

from explorer.display import display_messages

display_messages()

In [ ]:
#===================================================
# Cellule test
#===================================================
import inspect
import explorer.report

print(explorer.report.__file__)
print(inspect.signature(explorer.report.display_report))